In [3]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv("asr_evaluation_big_model.csv")

# Ensure text isn't truncated in the notebook view
pd.set_option('display.max_colwidth', None)

def print_thesis_test(dataframe, test_title, description, num_samples=3):
    print("=" * 90)
    print(f" TEST: {test_title}")
    print(f" DESCRIPTION: {description}")
    print("=" * 90)
    
    samples = dataframe.head(num_samples)
    if samples.empty:
        print(" [!] No samples met the filtering criteria for this test.")
        print("\n" + "-"*90 + "\n")
        return
        
    for idx, row in samples.iterrows():
        print(f"ID: {row['id']}")
        print(f"  [REF] Ground Truth : {row['ground_truth']}")
        print(f"  [RAW] Greedy Pred  : {row['greedy_pred']}")
        print(f"  [KNL] KenLM Pred   : {row['kenlm_pred']}")
        print(f"  [METRICS] Raw -> WER: {row['greedy_wer']:.2f} / CER: {row['greedy_cer']:.2f}")
        print(f"            KNL -> WER: {row['kenlm_wer']:.2f} / CER: {row['kenlm_cer']:.2f} (Delta WER: {row['wer_improvement']:.2f})")
        print(f"  [ALIGNMENTS] KenLM Errors -> Subs: {row['kenlm_substitutions']} | Ins: {row['kenlm_insertions']} | Del: {row['kenlm_deletions']}")
        print("-" * 90)
    print("\n" + "="*90 + "\n")

In [4]:
# 1. Highest Accuracy (WER+CER) Samples for the Raw Model
# We find where both metrics are absolutely 0.0, sorting by length to find impressive examples
highest_accuracy_raw = df[(df['greedy_wer'] == 0) & (df['greedy_cer'] == 0)].copy()
highest_accuracy_raw['len'] = highest_accuracy_raw['ground_truth'].apply(lambda x: len(str(x)))
highest_accuracy_raw = highest_accuracy_raw.sort_values(by='len', ascending=False)

print_thesis_test(
    highest_accuracy_raw, 
    "Highest Accuracy Raw Samples", 
    "Flawless (0% WER/CER) standalone acoustic predictions. Demonstrates upper bound of Wav2Vec2-Bert."
)

# 2. Overall Impact of the Language Model (Statistical Pivot)
total = len(df)
better = len(df[df['wer_improvement'] > 0])
worse = len(df[df['wer_improvement'] < 0])
same = len(df[df['wer_improvement'] == 0])

print("=" * 90)
print(" MACRO ANALYSIS: GLOBAL IMPACT OF THE LANGUAGE MODEL")
print("=" * 90)
print(f"Total Dataset Size              : {total} utterances")
print(f"KenLM Net Positive Improvements : {better} samples ({ (better/total)*100:.2f}%)")
print(f"KenLM Net Negative Degradations : {worse} samples ({ (worse/total)*100:.2f}%)")
print(f"Neutral / Unchanged Outputs     : {same} samples ({ (same/total)*100:.2f}%)")
print("=" * 90 + "\n")

 TEST: Highest Accuracy Raw Samples
 DESCRIPTION: Flawless (0% WER/CER) standalone acoustic predictions. Demonstrates upper bound of Wav2Vec2-Bert.


KeyError: 'id'

In [3]:
# 3. Instances where KenLM improved WER over the raw model
# Sorted by largest absolute drop in WER
kenlm_wins = df[df['wer_improvement'] > 0].sort_values(by='wer_improvement', ascending=False)

print_thesis_test(
    kenlm_wins, 
    "KenLM Improved WER", 
    "Cases where context corrected acoustic ambiguity (e.g., spelling fixes, homophones)."
)

# 4. Instances where KenLM degraded WER over the raw model
# Sorted by largest absolute increase in WER (most negative improvement value)
kenlm_losses = df[df['wer_improvement'] < 0].sort_values(by='wer_improvement', ascending=True)

print_thesis_test(
    kenlm_losses, 
    "KenLM Degraded WER", 
    "Cases where KenLM over-corrected correct acoustic signals or hallucinated common n-grams."
)

 TEST: KenLM Improved WER
 DESCRIPTION: Cases where context corrected acoustic ambiguity (e.g., spelling fixes, homophones).
ID: sample_284
  [REF] Ground Truth : żaqqtu nar tewqu nar kappellu nar kulma fuqu nar
  [RAW] Greedy Pred  : żaqtu ngħar tewqu ngħar kappellu ngħar kulma fuqu ngħar
  [KNL] KenLM Pred   : żaqtu nar tewqu nar kappellu nar kulma fuqu nar
  [METRICS] Raw -> WER: 0.56 / CER: 0.19
            KNL -> WER: 0.11 / CER: 0.02 (Delta WER: 0.44)
  [ALIGNMENTS] KenLM Errors -> Subs: 1 | Ins: 0 | Del: 0
------------------------------------------------------------------------------------------
ID: sample_959
  [REF] Ground Truth : x'outcomes qed jistenna mill-istess konferenza
  [RAW] Greedy Pred  : x'outchumbs qed jistenna amil l-istess konferenza
  [KNL] KenLM Pred   : x'outchumbs qed jistenna mill-istess konferenza
  [METRICS] Raw -> WER: 0.60 / CER: 0.11
            KNL -> WER: 0.20 / CER: 0.07 (Delta WER: 0.40)
  [ALIGNMENTS] KenLM Errors -> Subs: 1 | Ins: 0 | Del: 0
----

In [4]:
# 5. High CER but Low WER
# Scenario: The words are technically marked 'correct' by the evaluation tokenizer, 
# but inside the words are small character typos, or minor trailing suffix/prefix errors.
# We look for rows where WER is low (good) but CER is disproportionately high.
high_cer_low_wer = df[(df['kenlm_wer'] <= 0.15) & (df['kenlm_cer'] >= 0.15)].sort_values(by='kenlm_cer', ascending=False)

print_thesis_test(
    high_cer_low_wer, 
    "High CER but Low WER (KenLM Outputs)", 
    "Utterances with correct base words but frequent inner spelling typos, word compounding errors, or suffix issues."
)

# 6. High WER but Low CER
# Scenario: The transcription sounds incredibly close phonetically (low character difference), 
# but small shifts like splitting a word or replacing small structural words completely ruin the Word Error Rate.
# We look for rows where WER is poor, but CER remains low.
high_wer_low_cer = df[(df['kenlm_wer'] >= 0.40) & (df['kenlm_cer'] <= 0.15)].sort_values(by='kenlm_wer', ascending=False)

print_thesis_test(
    high_wer_low_cer, 
    "High WER but Low CER (KenLM Outputs)", 
    "Utterances where substitutions are phonetically near-flawless (e.g. 'cat' vs 'bat') or word-boundary splits broke entire word structures."
)

 TEST: High CER but Low WER (KenLM Outputs)
 DESCRIPTION: Utterances with correct base words but frequent inner spelling typos, word compounding errors, or suffix issues.
 [!] No samples met the filtering criteria for this test.

------------------------------------------------------------------------------------------

 TEST: High WER but Low CER (KenLM Outputs)
 DESCRIPTION: Utterances where substitutions are phonetically near-flawless (e.g. 'cat' vs 'bat') or word-boundary splits broke entire word structures.
ID: sample_1511
  [REF] Ground Truth : tama għall-afrikani
  [RAW] Greedy Pred  : tama għal l-afrikani
  [KNL] KenLM Pred   : tama għal l-afrikani
  [METRICS] Raw -> WER: 1.00 / CER: 0.05
            KNL -> WER: 1.00 / CER: 0.05 (Delta WER: 0.00)
  [ALIGNMENTS] KenLM Errors -> Subs: 1 | Ins: 1 | Del: 0
------------------------------------------------------------------------------------------
ID: sample_1556
  [REF] Ground Truth : smartcity rridu nagħmluha taħdem
  [RAW] Greedy 

In [5]:
# =====================================================================
# ADVANCED TEST A: Word Truncation / Dropped Text (Beta Diagnostic)
# =====================================================================
# Find where KenLM predicted significantly fewer words than the raw baseline
df['raw_wcount'] = df['greedy_pred'].apply(lambda x: len(str(x).split()))
df['knl_wcount'] = df['kenlm_pred'].apply(lambda x: len(str(x).split()))
df['dropped_words'] = df['raw_wcount'] - df['knl_wcount']

# Look for cases where KenLM dropped 3 or more words that the acoustic model caught
text_truncations = df[df['dropped_words'] >= 3].sort_values(by='dropped_words', ascending=False)

print_thesis_test(
    text_truncations, 
    "KenLM Text Truncations (Dropped Words)", 
    "Cases where KenLM aggressively deleted words. Highly useful for discussing Beta penalty optimization."
)

# =====================================================================
# ADVANCED TEST B: Length Bias & Short Utterance Vulnerability
# =====================================================================
# Language models notoriously struggle with 1-2 word utterances because there is zero context.
# Let's isolate very short sentences to see if KenLM broke them.
short_sentences = df[df['ground_truth'].apply(lambda x: len(str(x).split())) <= 2]
short_failures = short_sentences[short_sentences['wer_improvement'] < 0].sort_values(by='wer_improvement')

print_thesis_test(
    short_failures, 
    "Short Utterance Vulnerability (1-2 Words)", 
    "Analyzes how the language model behaves when deprived of structural/contextual history."
)

 TEST: KenLM Text Truncations (Dropped Words)
 DESCRIPTION: Cases where KenLM aggressively deleted words. Highly useful for discussing Beta penalty optimization.
ID: sample_164
  [REF] Ground Truth : ħafna jiġu assoċjati mal-viljakk u dan jista' juri li l-ħrejjef xi kultant ikun għandhom huma ra-
  [RAW] Greedy Pred  : ħafna jiġu assoċjati mal-vilja u dan jista' juri li il-ħrejjef xi kultant ikun għandhom huma ra
  [KNL] KenLM Pred   : ħafna jiġu assoċjati mal-viljaan jista' juri li l-ħrejjef xi kultant ikun għandhom huma
  [METRICS] Raw -> WER: 0.19 / CER: 0.04
            KNL -> WER: 0.25 / CER: 0.10 (Delta WER: -0.06)
  [ALIGNMENTS] KenLM Errors -> Subs: 1 | Ins: 0 | Del: 3
------------------------------------------------------------------------------------------


 TEST: Short Utterance Vulnerability (1-2 Words)
 DESCRIPTION: Analyzes how the language model behaves when deprived of structural/contextual history.
 [!] No samples met the filtering criteria for this test.

-----------

In [5]:
import pandas as pd
import numpy as np

# Load the updated data
df = pd.read_csv("asr_evaluation_big_model.csv")

# Ensure text isn't truncated in the notebook view
pd.set_option('display.max_colwidth', None)

def print_thesis_test(dataframe, test_title, description, num_samples=3):
    print("=" * 90)
    print(f" TEST: {test_title}")
    print(f" DESCRIPTION: {description}")
    print("=" * 90)
    
    samples = dataframe.head(num_samples)
    if samples.empty:
        print(" [!] No samples met the filtering criteria for this test.")
        print("\n" + "-"*90 + "\n")
        return
        
    for idx, row in samples.iterrows():
        # FIX: Changed row['id'] to row['filename'] to match your updated schema
        print(f"FILE/ID: {row['filename']}") 
        print(f"  SOURCE:  [{row['source_domain']}]") # Added source domain tracking visibility
        print(f"  [REF] Ground Truth : {row['ground_truth']}")
        print(f"  [RAW] Greedy Pred  : {row['greedy_pred']}")
        print(f"  [KNL] KenLM Pred   : {row['kenlm_pred']}")
        print(f"  [METRICS] Raw -> WER: {row['greedy_wer']:.2f} / CER: {row['greedy_cer']:.2f}")
        print(f"            KNL -> WER: {row['kenlm_wer']:.2f} / CER: {row['kenlm_cer']:.2f} (Delta WER: {row['wer_improvement']:.2f})")
        print(f"  [ALIGNMENTS] KenLM Errors -> Subs: {row['kenlm_substitutions']} | Ins: {row['kenlm_insertions']} | Del: {row['kenlm_deletions']}")
        print("-" * 90)
    print("\n" + "="*90 + "\n")

# =====================================================================
# ANALYSIS 1: Highest Accuracy (WER+CER) Samples for the Raw Model
# =====================================================================
highest_accuracy_raw = df[(df['greedy_wer'] == 0) & (df['greedy_cer'] == 0)].copy()
highest_accuracy_raw['len'] = highest_accuracy_raw['ground_truth'].apply(lambda x: len(str(x)))
highest_accuracy_raw = highest_accuracy_raw.sort_values(by='len', ascending=False)

print_thesis_test(
    highest_accuracy_raw, 
    "Highest Accuracy Raw Samples", 
    "Flawless (0% WER/CER) standalone acoustic predictions. Demonstrates upper bound of Wav2Vec2-Bert."
)

# =====================================================================
# ANALYSIS 2: Macro Analysis (Global Impact of LM)
# =====================================================================
total = len(df)
better = len(df[df['wer_improvement'] > 0])
worse = len(df[df['wer_improvement'] < 0])
same = len(df[df['wer_improvement'] == 0])

print("=" * 90)
print(" MACRO ANALYSIS: GLOBAL IMPACT OF THE LANGUAGE MODEL")
print("=" * 90)
print(f"Total Dataset Size              : {total} utterances")
print(f"KenLM Net Positive Improvements : {better} samples ({ (better/total)*100:.2f}%)")
print(f"KenLM Net Negative Degradations : {worse} samples ({ (worse/total)*100:.2f}%)")
print(f"Neutral / Unchanged Outputs     : {same} samples ({ (same/total)*100:.2f}%)")
print("=" * 90 + "\n")


# =====================================================================
# NEW ANALYSIS 3: Thesis Subset Breakdown (MASRI vs. CommonVoice)
# =====================================================================
print("=" * 90)
print(" SUBSET ANALYSIS: DOMAIN SPECIFIC PERFORMANCE PIVOT")
print("=" * 90)

# Build descriptive pivot aggregations grouped by your new 'source_domain' column
domain_pivot = df.groupby('source_domain').agg(
    total_utterances=('filename', 'count'),
    avg_raw_wer=('greedy_wer', 'mean'),
    avg_kenlm_wer=('kenlm_wer', 'mean'),
    avg_wer_delta=('wer_improvement', 'mean'),
    total_substitutions=('kenlm_substitutions', 'sum'),
    total_insertions=('kenlm_insertions', 'sum'),
    total_deletions=('kenlm_deletions', 'sum')
).reset_index()

# Display formatted table for your thesis draft notebook
display(domain_pivot.round(4))
print("=" * 90 + "\n")


# =====================================================================
# NEW ANALYSIS 4: Isolated Domain Case-Study Grabber
# =====================================================================
# Extract specific instances where KenLM significantly helped CommonVoice for your text examples
cv_wins = df[(df['source_domain'] == 'CommonVoice') & (df['wer_improvement'] > 0.15)].copy()
cv_wins = cv_wins.sort_values(by='wer_improvement', ascending=False)

print_thesis_test(
    cv_wins,
    "CommonVoice Distinctive KenLM Wins",
    "Samples where KenLM fixed corrupted acoustic outputs on CommonVoice data specifically by > 15% WER.",
    num_samples=2
)

 TEST: Highest Accuracy Raw Samples
 DESCRIPTION: Flawless (0% WER/CER) standalone acoustic predictions. Demonstrates upper bound of Wav2Vec2-Bert.
FILE/ID: MSRDV_M_08_DV_00046
  SOURCE:  [MASRI]
  [REF] Ground Truth : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [RAW] Greedy Pred  : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [KNL] KenLM Pred   : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [METRICS] Raw -> WER: 0.00 / CER: 0.00
            KNL -> WER: 0.00 / CER: 0.00 (Delta WER: 0.00)
  [ALIGNMENTS] KenLM Errors -> Subs: 0 | Ins: 0 | Del: 0
------------------------------------------------------------------------------------------
FILE/ID: MSRDV_M_08_DV_00094
  SOURCE:  [MASRI]
  [REF] Ground Truth : ikollhom

,source_domain,total_utterances,avg_raw_wer,avg_kenlm_wer,avg_wer_delta,total_substitutions,total_insertions,total_deletions
0,CV,1235,0.0478,0.0359,0.0119,258,18,12
1,MASRI,648,0.2025,0.1828,0.0196,1051,211,235



 TEST: CommonVoice Distinctive KenLM Wins
 DESCRIPTION: Samples where KenLM fixed corrupted acoustic outputs on CommonVoice data specifically by > 15% WER.
 [!] No samples met the filtering criteria for this test.

------------------------------------------------------------------------------------------



### Tests

In [7]:
import pandas as pd
import numpy as np

# Load the updated data
df = pd.read_csv("asr_evaluation_big_model.csv")

# Ensure text isn't truncated in the notebook view
pd.set_option('display.max_colwidth', None)

def print_thesis_test(dataframe, test_title, description, num_samples=3):
    print("=" * 90)
    print(f" TEST: {test_title}")
    print(f" DESCRIPTION: {description}")
    print("=" * 90)
    
    samples = dataframe.head(num_samples)
    if samples.empty:
        print(" [!] No samples met the filtering criteria for this test.")
        print("\n" + "-"*90 + "\n")
        return
        
    for idx, row in samples.iterrows():
        print(f"FILE/ID: {row['filename']}") 
        print(f"  SOURCE:  [{row['source_domain']}]") 
        print(f"  [REF] Ground Truth : {row['ground_truth']}")
        print(f"  [RAW] Greedy Pred  : {row['greedy_pred']}")
        print(f"  [KNL] KenLM Pred   : {row['kenlm_pred']}")
        print(f"  [METRICS] Raw -> WER: {row['greedy_wer']:.2f} / CER: {row['greedy_cer']:.2f}")
        print(f"            KNL -> WER: {row['kenlm_wer']:.2f} / CER: {row['kenlm_cer']:.2f} (Delta WER: {row['wer_improvement']:.2f})")
        print(f"  [ALIGNMENTS] KenLM Errors -> Subs: {row['kenlm_substitutions']} | Ins: {row['kenlm_insertions']} | Del: {row['kenlm_deletions']}")
        print("-" * 90)
    print("\n" + "="*90 + "\n")

# =====================================================================
# ANALYSIS 1: Highest Accuracy (WER+CER) Samples for the Raw Model
# =====================================================================
highest_accuracy_raw = df[(df['greedy_wer'] == 0) & (df['greedy_cer'] == 0)].copy()
highest_accuracy_raw['len'] = highest_accuracy_raw['ground_truth'].apply(lambda x: len(str(x)))
highest_accuracy_raw = highest_accuracy_raw.sort_values(by='len', ascending=False)

print_thesis_test(
    highest_accuracy_raw, 
    "Highest Accuracy Raw Samples", 
    "Flawless (0% WER/CER) standalone acoustic predictions. Demonstrates upper bound of Wav2Vec2-Bert."
)

# =====================================================================
# ANALYSIS 2: Macro Analysis (Global Impact of LM)
# =====================================================================
total = len(df)
better = len(df[df['wer_improvement'] > 0])
worse = len(df[df['wer_improvement'] < 0])
same = len(df[df['wer_improvement'] == 0])

print("=" * 90)
print(" MACRO ANALYSIS: GLOBAL IMPACT OF THE LANGUAGE MODEL")
print("=" * 90)
print(f"Total Dataset Size              : {total} utterances")
print(f"KenLM Net Positive Improvements : {better} samples ({ (better/total)*100:.2f}%)")
print(f"KenLM Net Negative Degradations : {worse} samples ({ (worse/total)*100:.2f}%)")
print(f"Neutral / Unchanged Outputs     : {same} samples ({ (same/total)*100:.2f}%)")
print("=" * 90 + "\n")

# =====================================================================
# ANALYSIS 3: Thesis Subset Breakdown (MASRI vs. CommonVoice)
# =====================================================================
print("=" * 90)
print(" SUBSET ANALYSIS: DOMAIN SPECIFIC PERFORMANCE PIVOT")
print("=" * 90)

domain_pivot = df.groupby('source_domain').agg(
    total_utterances=('filename', 'count'),
    avg_raw_wer=('greedy_wer', 'mean'),
    avg_kenlm_wer=('kenlm_wer', 'mean'),
    avg_wer_delta=('wer_improvement', 'mean'),
    total_substitutions=('kenlm_substitutions', 'sum'),
    total_insertions=('kenlm_insertions', 'sum'),
    total_deletions=('kenlm_deletions', 'sum')
).reset_index()

display(domain_pivot.round(4))
print("=" * 90 + "\n")

# =====================================================================
# ANALYSIS 4: Isolated Domain Case-Study Grabber
# =====================================================================
cv_wins = df[(df['source_domain'] == 'CommonVoice') & (df['wer_improvement'] > 0.15)].copy()
cv_wins = cv_wins.sort_values(by='wer_improvement', ascending=False)

print_thesis_test(
    cv_wins,
    "CommonVoice Distinctive KenLM Wins",
    "Samples where KenLM fixed corrupted acoustic outputs on CommonVoice data specifically by > 15% WER.",
    num_samples=2
)

# =====================================================================
# INTEGRATED LINGUISTIC AND ALGORITHMIC QUALITATIVE ANALYSIS TESTS
# =====================================================================

# 5. Core Language Model Wins (Contextual & Grammatical Corrections)
# Looks for instances where the acoustic model was messy but KenLM salvaged it
kenlm_macro_wins = df[df['wer_improvement'] > 0.25].sort_values(by='wer_improvement', ascending=False)

print_thesis_test(
    kenlm_macro_wins,
    "Significant Language Model Wins (Linguistic Reconstruction)",
    "Acoustically flawed predictions successfully resolved by KenLM context constraints.\n"
    "Look for: Restorations of orthographically silent letters (għ, h), correct gemination, and grammar particles."
)

# 6. Language Model Failures (Over-Correction / Search-Space Distortions)
# Isolates instances where KenLM ruined an otherwise clean or accurate greedy string
kenlm_macro_fails = df[df['wer_improvement'] < -0.15].sort_values(by='wer_improvement', ascending=True)

print_thesis_test(
    kenlm_macro_fails,
    "Language Model Degradations (Over-Correction)",
    "Instances where KenLM overwrote acoustically sound character paths.\n"
    "Look for: Out-of-Vocabulary (OOV) names/surnames, code-switching to English, or low-frequency Maltese words forced into high-frequency N-grams."
)

# 7. Acoustic Acoustic Dropouts (High Deletions Analysis)
# Captures instances where Wav2Vec2-Bert drops phones or skips short grammatical text components
high_deletions = df[df['greedy_deletions'] >= 3].sort_values(by='greedy_deletions', ascending=False)

print_thesis_test(
    high_deletions,
    "Severe Acoustic Deletions (Raw Model Dropout)",
    "Utterances where the raw model missed major acoustic details or dropped short grammatical tokens entirely.\n"
    "Look for: Fast-spoken speech boundaries, background noise, or missing proclitic clitics (e.g., b', t', ta')."
)

# 8. High Substitution Rate Analysis (Phonetic vs. Orthographic Confusions)
# Catches severe token mismatches to highlight phonetic approximations
high_substitutions = df[df['kenlm_substitutions'] >= 4].sort_values(by='kenlm_substitutions', ascending=False)

print_thesis_test(
    high_substitutions,
    "High Word Substitutions Case Studies",
    "Sentences indicating fundamental vocabulary mapping collisions.\n"
    "Look for: Homophones, dialect discrepancies, or cases where the acoustic model mapped phonetic sequences to a totally different lexical semantic word."
)

 TEST: Highest Accuracy Raw Samples
 DESCRIPTION: Flawless (0% WER/CER) standalone acoustic predictions. Demonstrates upper bound of Wav2Vec2-Bert.
FILE/ID: MSRDV_M_08_DV_00046
  SOURCE:  [MASRI]
  [REF] Ground Truth : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [RAW] Greedy Pred  : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [KNL] KenLM Pred   : tista' titgħallem it-taljan tista' titgħallem il-franċiż tista' titgħallem l-ispanjol tista' titgħallem il-ġermaniż dak ukoll f' livell
  [METRICS] Raw -> WER: 0.00 / CER: 0.00
            KNL -> WER: 0.00 / CER: 0.00 (Delta WER: 0.00)
  [ALIGNMENTS] KenLM Errors -> Subs: 0 | Ins: 0 | Del: 0
------------------------------------------------------------------------------------------
FILE/ID: MSRDV_M_08_DV_00094
  SOURCE:  [MASRI]
  [REF] Ground Truth : ikollhom

,source_domain,total_utterances,avg_raw_wer,avg_kenlm_wer,avg_wer_delta,total_substitutions,total_insertions,total_deletions
0,CV,1235,0.0478,0.0359,0.0119,258,18,12
1,MASRI,648,0.2025,0.1828,0.0196,1051,211,235



 TEST: CommonVoice Distinctive KenLM Wins
 DESCRIPTION: Samples where KenLM fixed corrupted acoustic outputs on CommonVoice data specifically by > 15% WER.
 [!] No samples met the filtering criteria for this test.

------------------------------------------------------------------------------------------

 TEST: Significant Language Model Wins (Linguistic Reconstruction)
 DESCRIPTION: Acoustically flawed predictions successfully resolved by KenLM context constraints.
Look for: Restorations of orthographically silent letters (għ, h), correct gemination, and grammar particles.
FILE/ID: MSRDV_F_08_DV_00003
  SOURCE:  [MASRI]
  [REF] Ground Truth : żaqqtu nar tewqu nar kappellu nar kulma fuqu nar
  [RAW] Greedy Pred  : żaqtu ngħar tewqu ngħar kappellu ngħar kulma fuqu ngħar
  [KNL] KenLM Pred   : żaqtu nar tewqu nar kappellu nar kulma fuqu nar
  [METRICS] Raw -> WER: 0.56 / CER: 0.19
            KNL -> WER: 0.11 / CER: 0.02 (Delta WER: 0.44)
  [ALIGNMENTS] KenLM Errors -> Subs: 1 | Ins: 0

In [1]:
def run_qualitative_analysis(df, output_report_path="qualitative_report.md"):
    """
    Slices the inference dataframe by domains, tracks 100% correct (Zero-WER) metrics,
    and extracts robust behavioral profiles for dissertation write-ups.
    """
    print("\n--- Running Comprehensive Qualitative Analysis ---")
    
    # Pre-calculate composite metrics
    df['kenlm_avg_error'] = (df['kenlm_wer'] + df['kenlm_cer']) / 2.0
    
    domains = df['source_domain'].unique()
    markdown_report = "# 📊 Qualitative Error Analysis & Diagnostic Report\n\n"
    
    for domain in domains:
        df_dom = df[df['source_domain'] == domain].copy()
        total_samples = len(df_dom)
        mean_domain_wer = df_dom['kenlm_wer'].mean()
        
        # --- 100% Correct (Zero-WER) Calculations ---
        greedy_perfect = (df_dom['greedy_wer'] == 0).sum()
        kenlm_perfect = (df_dom['kenlm_wer'] == 0).sum()
        
        greedy_perfect_pct = (greedy_perfect / total_samples) * 100
        kenlm_perfect_pct = (kenlm_perfect / total_samples) * 100
        
        # Specific decoder interactions
        rescued_by_kenlm = df_dom[(df_dom['greedy_wer'] > 0) & (df_dom['kenlm_wer'] == 0)]
        damaged_by_kenlm = df_dom[(df_dom['greedy_wer'] == 0) & (df_dom['kenlm_wer'] > 0)]
        
        # --- Build Domain Header & Quantitative Summary ---
        markdown_report += f"## 🌍 Test Set / Domain: {domain}\n"
        markdown_report += f"*Total samples: {total_samples} | Domain Mean KenLM WER: {mean_domain_wer:.4f}*\n\n"
        
        markdown_report += "### 🎯 Perfect Transcription (Zero-WER) Summary\n"
        markdown_report += "| Decoder | Perfect Sentences (Count) | Perfect Sentences (%) |\n"
        markdown_report += "| :--- | :---: | :---: |\n"
        markdown_report += f"| **Greedy Baseline** | {greedy_perfect} / {total_samples} | {greedy_perfect_pct:.2f}% |\n"
        markdown_report += f"| **With KenLM** | {kenlm_perfect} / {total_samples} | {kenlm_perfect_pct:.2f}% |\n\n"
        markdown_report += f"* KenLM successfully **rescued** `{len(rescued_by_kenlm)}` error-prone greedy sentences to 100% accuracy.\n"
        markdown_report += f"* KenLM accidentally **damaged** `{len(damaged_by_kenlm)}` previously perfect sentences.\n\n"
        
        # --- Structured Profiles for Write-up Selection ---
        profiles = {
            "🏆 1. Best Performance Pool (Top candidates for 'Perfect Baseline Case')": 
                df_dom.nsmallest(10, 'kenlm_avg_error'),
                
            "💥 2. Worst Performance Pool (Top candidates for 'Catastrophe Case')": 
                df_dom.nlargest(3, 'kenlm_avg_error'),
                
            "📉 3. Mean Statistical Samples (Closest to Domain Mean WER)": 
                df_dom.iloc[(df_dom['kenlm_wer'] - mean_domain_wer).abs().argsort()[:3]],
                
            "✨ 4. KenLM Rescues (Acoustic typos successfully corrected by language model)": 
                rescued_by_kenlm.head(3),
                
            "💔 5. KenLM Regressions (Linguistic over-corrections that ruined perfect text)": 
                damaged_by_kenlm.sort_values('wer_improvement', ascending=True).head(10),
                
            "🔍 6. High CER but Low WER (Phonetic shifts / Orthographic variations)": 
                df_dom[(df_dom['kenlm_cer'] > 0.25) & (df_dom['kenlm_wer'] < 0.15)].head(3),
                
            "🧠 7. Acoustic Triumph, Decoder Failure (Overly punitive word matching - Low CER / High WER)": 
                df_dom[(df_dom['kenlm_cer'] < 0.08) & (df_dom['kenlm_wer'] > 0.35)].head(3)
        }
        
        # Format profiles into Markdown
        for title, sampled_df in profiles.items():
            markdown_report += f"### {title}\n"
            if sampled_df.empty:
                markdown_report += "_No samples found matching these specific mathematical bounds in this domain._\n\n"
                continue
                
            for idx, (_, row) in enumerate(sampled_df.iterrows(), 1):
                markdown_report += f"**Sample #{idx} | File:** `{row['filename']}`\n"
                markdown_report += f"* **Ground Truth:** \"{row['ground_truth']}\"\n"
                markdown_report += f"* **Greedy Pred:** \"{row['greedy_pred']}\" (WER: {row['greedy_wer']:.2f}, CER: {row['greedy_cer']:.2f})\n"
                markdown_report += f"* **KenLM Pred:** \"{row['kenlm_pred']}\" (WER: {row['kenlm_wer']:.2f}, CER: {row['kenlm_cer']:.2f})\n"
                markdown_report += f"* **Metrics:** Δ WER Improvement: {row['wer_improvement']:.2f} | Error Composite: {row['kenlm_avg_error']:.2f}\n"
                markdown_report += f"  * Total Edits -> Subs: {row['kenlm_substitutions']}, Ins: {row['kenlm_insertions']}, Del: {row['kenlm_deletions']}\n\n"
        
        markdown_report += "---\n\n"
        
    # Write report to disk
    with open(output_report_path, "w", encoding="utf-8") as f:
        f.write(markdown_report)
        
    print(f"🔬 Comprehensive diagnostic report generated successfully at: {output_report_path}")

In [2]:
if __name__ == "__main__":
    # Load the saved CSV file back into a DataFrame object
    df_results = pd.read_csv("asr_evaluation_big_model.csv")
    
    # Run the qualitative analysis on it
    run_qualitative_analysis(df_results, output_report_path="qualitative_report.md")

NameError: name 'pd' is not defined